
# Analisis Hubungan Antar Variabel — Skrining Sleman 2025

Notebook ini menjawab satu pertanyaan:

> **Hubungan apa yang ada antara karakteristik seseorang dan risiko penyakitnya, dan
> seberapa kuat hubungan itu?**

Analisisnya murni **deskriptif** — mengukur dan menggambarkan hubungan. Tidak ada pemodelan
di sini; itu ada di `Script penyakitan/Stacked Revisi.ipynb`.

## Susunan notebook

Notebook disusun **menurut penyakit**, bukan menurut variabel. Tiap penyakit mendapat satu
profil lengkap berisi seluruh variabel bebasnya sekaligus, sehingga pertanyaan *"apa yang
menggerakkan penyakit ini"* terjawab dalam satu tampilan.

| Bagian | Isi |
|---|---|
| 1 | Data & fungsi bantu |
| 2 | **Cara membaca grafik risiko** — kekeliruan yang mudah terjadi |
| 3 | **Hubungan antar variabel bebas** — sebelum masuk ke penyakit |
| 4 | **Diabetes Melitus** |
| 5 | **Hipertensi** |
| 6 | **Kanker Paru** |
| 7 | **TBC** |
| 8 | **Segmen** — peta panas seluruh penyakit |
| 9 | Ringkasan lintas penyakit |
| 10 | Korelasi bukan sebab-akibat |

## Variabel yang dianalisis

| Variabel bebas | Tipe |
|---|---|
| `Jeniskelamin` | kategorik biner (L/P) |
| `Usia` | numerik |
| `Tinggibadan` | numerik |
| `Berat Badan` | numerik |
| `IMT_recalculated` | numerik |
| `segmen` | kategorik nominal (7 kategori) |

Empat target risiko: Diabetes Melitus, Hipertensi, Kanker Paru, TBC — semuanya **biner**.

## Metode: satu ukuran untuk tiap pasangan tipe

Tidak semua pasangan variabel bisa diukur dengan korelasi Pearson. Pearson mengasumsikan dua
variabel numerik dengan hubungan linear — memakainya pada `segmen` (yang tidak punya urutan)
akan menghasilkan angka yang tidak bermakna. Karena itu tiap pasangan memakai ukuran yang
sesuai:

| Pasangan | Ukuran | Rentang | Arti |
|---|---|---|---|
| numerik ↔ numerik | **Pearson** *r* | −1 … +1 | kekuatan hubungan **linear** |
| numerik ↔ numerik | **Spearman** ρ | −1 … +1 | kekuatan hubungan **monoton** (tahan outlier) |
| numerik ↔ target biner | **point-biserial** *r* | −1 … +1 | sama dengan Pearson memakai target 0/1 |
| biner ↔ biner | **phi** (φ) | 0 … 1 | kasus khusus Cramér's V untuk tabel 2×2 |
| nominal ↔ biner | **Cramér's V** | 0 … 1 | kekuatan asosiasi, tanpa arah |
| nominal ↔ numerik | **rasio korelasi** (η) | 0 … 1 | seberapa besar rata-rata berbeda antar kategori |

Ukuran tanpa arah (V dan η) tidak bisa negatif — ia hanya menyatakan **seberapa kuat**, bukan
ke arah mana.

## Panduan membaca besaran

| |nilai| | Tafsiran |
|---|---|
| < 0.10 | dapat diabaikan |
| 0.10 – 0.30 | lemah |
| 0.30 – 0.50 | sedang |
| > 0.50 | kuat |

## Kenapa notebook ini tidak memuat nilai-p

Dengan **410.398 baris**, hampir semua hubungan akan "signifikan secara statistik" — bahkan
korelasi sebesar 0.01 pun menghasilkan p < 0.001. Nilai-p pada ukuran sampel sebesar ini
hanya memberi tahu bahwa datanya banyak, bukan bahwa hubungannya penting.

Karena itu notebook ini melaporkan **besaran efek** (*effect size*), bukan signifikansi.

## Yang sudah ada di notebook lain (tidak diulang di sini)

- Korelasi **antar target** (komorbiditas) — `Analisis Data Bersih` §B4
- Prevalensi per kategori BMI — `Klasifikasi BMI` §7
- Prevalensi per subkelompok usia/jenis kelamin/segmen — `Analisis Data Bersih` §B3
- *Feature importance* berbasis model — `Stacked Revisi` §7


---
# 1. Data & Fungsi Bantu

In [ ]:

# ===================== IMPORTS & DATA =====================
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Modul bersama + konfigurasi wilayah. Akar proyek dicari dari folder kerja,
# jadi tidak ada satu pun jalur absolut di notebook ini: seluruh folder boleh
# dipindah atau diserahkan ke orang lain tanpa sel ini perlu diubah.
WILAYAH = "Sleman"

AKAR = Path.cwd()
while not (AKAR / "Bersama").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "Bersama"))

import konfigurasi as cfg      # jalur + cara baca berkas mentah tiap wilayah
import viz_style as vz

vz.apply_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 190)

DATA_DIR = cfg.folder_data(WILAYAH)

df = pd.read_csv(os.path.join(DATA_DIR, "dataset_master.csv"))

NUM = ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]
CAT = ["Jeniskelamin", "segmen"]

TARGETS = {
    "Diabetes": ("Berisiko Diabetes Melitus", "15+"),
    "Hipertensi": ("Berisiko Hipertensi", "15+"),
    "Kanker Paru": ("Berisiko Kanker Paru", "45+"),
    "TBC": ("Berisiko TBC", "semua usia"),
}

print(f"Data  : {len(df):,} catatan skrining")
print(f"Usia  : {int(df.Usia.min())} - {int(df.Usia.max())} tahun")
display(df[NUM + CAT].head())


## Dua aturan yang berlaku di seluruh notebook

**1. Tiap target hanya boleh dihitung pada populasi yang berhak ditanyai.** Label
`Tidak Dapat Pertanyaan` berarti responden **tidak memenuhi syarat usia**, bukan datanya
hilang. Diabetes & Hipertensi hanya ditanyakan pada usia 15+, Kanker Paru 45+, TBC semua
usia. Menghitung korelasi tanpa menyaring ini akan mencampur "tidak berisiko" dengan
"tidak ditanya".

**2. Kekuatan hubungan antar penyakit tidak sepenuhnya sebanding.** Karena populasinya
berbeda, angka untuk Kanker Paru (hanya 45+) tidak bisa dibandingkan mentah-mentah dengan
TBC (semua usia). Ini diingatkan lagi di §9.

In [ ]:

# ===================== FUNGSI UKURAN ASOSIASI =====================
def eligible(nama):
    # Kembalikan baris yang memang ditanyai untuk target ini, beserta targetnya sebagai 0/1.
    col, _ = TARGETS[nama]
    sub = df[df[col].isin(["Berisiko", "Tidak Berisiko"])].copy()
    return sub, (sub[col] == "Berisiko").astype(int)


def cramers_v(x, y):
    # Kekuatan asosiasi antara dua variabel kategorik (0..1, tanpa arah).
    # Untuk tabel 2x2 nilainya sama dengan koefisien phi.
    ct = pd.crosstab(x, y)
    if min(ct.shape) < 2:
        return np.nan
    n = ct.to_numpy().sum()
    exp = np.outer(ct.sum(axis=1), ct.sum(axis=0)) / n
    chi2 = ((ct.to_numpy() - exp) ** 2 / exp).sum()
    return float(np.sqrt((chi2 / n) / (min(ct.shape) - 1)))


def correlation_ratio(kategori, nilai):
    # Rasio korelasi eta (0..1): akar dari proporsi ragam 'nilai' yang dijelaskan
    # oleh perbedaan rata-rata antar kategori.
    grand = nilai.mean()
    g = nilai.groupby(kategori, observed=True)
    antar = (g.count() * (g.mean() - grand) ** 2).sum()
    total = ((nilai - grand) ** 2).sum()
    return float(np.sqrt(antar / total)) if total > 0 else np.nan


def tafsir(v):
    a = abs(v)
    if np.isnan(a):
        return "-"
    if a < 0.10:
        return "diabaikan"
    if a < 0.30:
        return "lemah"
    if a < 0.50:
        return "sedang"
    return "kuat"


print("Fungsi siap: eligible(), cramers_v(), correlation_ratio(), tafsir()")


---
# 2. Cara Membaca Grafik Risiko

Sebelum masuk ke penyakit, satu kekeliruan perlu dijelaskan lebih dulu karena sangat mudah
terjadi dan membalik kesimpulan.

Cara yang terlihat masuk akal tetapi **keliru** adalah menyaring hanya orang yang berisiko,
lalu membuat histogram usianya. Tinggi tiap batang pada grafik semacam itu adalah:

```
jumlah berisiko  =  jumlah peserta seusia itu  x  risiko pada usia itu
```

Faktor pertama biasanya mendominasi, sehingga yang tergambar sebenarnya adalah **demografi
peserta skrining**, bukan risiko. Tiga panel berikut membedah persoalannya memakai TBC —
target yang rentang usianya paling lebar sehingga efeknya paling jelas.

In [ ]:

# ===================== JUMLAH vs PREVALENSI: KENAPA HITUNGAN MENYESATKAN =====================
sub_t, y_t = eligible("TBC")
bt = pd.cut(sub_t["Usia"], [0, 15, 25, 35, 45, 55, 65, 75, 120], right=False,
            labels=["<15", "15-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"])

ringkas_t = pd.DataFrame({
    "Diskrining": y_t.groupby(bt, observed=True).size(),
    "Berisiko": y_t.groupby(bt, observed=True).sum(),
})
ringkas_t["Prevalensi (%)"] = (ringkas_t["Berisiko"] / ringkas_t["Diskrining"] * 100).round(2)

fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))
x = ringkas_t.index.astype(str)

axes[0].bar(x, ringkas_t["Diskrining"], color="#9aa6b2")
axes[0].set_title("1. Berapa banyak yang DISKRINING\n(penyebut)")
axes[0].set_ylabel("Jumlah peserta")
vz.thousands(axes[0])

axes[1].bar(x, ringkas_t["Berisiko"], color="#c0504d")
axes[1].set_title("2. KELIRU — jumlah yang berisiko\n(panel 1 x panel 3)")
axes[1].set_ylabel("Jumlah orang berisiko")
vz.thousands(axes[1])

axes[2].bar(x, ringkas_t["Prevalensi (%)"], color="#2d8746")
axes[2].set_title("3. BENAR — persen yang berisiko\n(ukuran risiko sesungguhnya)")
axes[2].set_ylabel("% Berisiko")

# Sorot dua kelompok yang ceritanya berbalik.
for ax, kol in [(axes[1], "Berisiko"), (axes[2], "Prevalensi (%)")]:
    for lbl in ["15-24", "75+"]:
        i = list(x).index(lbl)
        ax.patches[i].set_edgecolor("black")
        ax.patches[i].set_linewidth(2.5)
        v = ringkas_t.loc[lbl, kol]
        ax.annotate(f"{v:,.2f}" if kol.startswith("Prev") else f"{v:,}",
                    (i, v), ha="center", va="bottom", fontsize=12, weight="bold")

for ax in axes:
    ax.set_xlabel("Kelompok usia")
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("TBC — kenapa histogram 'hanya yang berisiko' bukan ukuran risiko", y=1.00)
plt.tight_layout()
plt.show()

display(ringkas_t)


### Bandingkan dua kelompok yang ditandai kotak hitam

| Kelompok | Diskrining | Berisiko | Prevalensi |
|---|---|---|---|
| **<15** | 62.227 | **2.052** | **3,30%** |
| **75+** | 15.048 | **1.528** | **10,15%** |

Pada panel 2 batang kelompok `<15` **lebih tinggi** daripada batang `75+` — seolah anak-anak
lebih bermasalah. Padahal kelompok 75+ **3,1 kali lebih berisiko**. Batangnya rendah
semata-mata karena lansia yang ikut skrining memang sedikit: 15.048 orang melawan 62.227.

Kesalahan yang lebih berbahaya ada di ujung kanan panel 2: jumlah kasus **memuncak di 55–64
(6.657) lalu menurun** menjadi 4.821 pada 65–74, sehingga pembaca akan menyimpulkan *"risiko
TBC menurun setelah usia 65"*. Panel 3 menunjukkan kebalikannya — prevalensi justru **naik**
dari 11,70% ke **12,40%**, dan puncaknya ada di 65–74, bukan di 55–64.

### Akibatnya untuk susunan grafik di bagian §4–§7

Tiap penyakit digambarkan dengan **dua baris grafik yang saling melengkapi**:

| Baris | Bentuk | Menjawab |
|---|---|---|
| **Atas** | histogram **densitas**, dua kurva (Berisiko vs Tidak Berisiko) | *"di mana orangnya berada"* |
| **Bawah** | garis prevalensi per desil | *"seberapa besar risikonya"* |

Histogram di baris atas memakai **densitas**, bukan hitungan, supaya kedua kelompok bisa
dibandingkan meski jumlahnya sangat berbeda. Jarak antara kedua kurva menunjukkan seberapa
kuat variabel itu memisahkan risiko.

Kolomnya sejajar, jadi bisa dibaca ke bawah: *"banyak orang di rentang ini, dan risikonya
sekian"*.


---
# 3. Hubungan Antar Variabel Bebas

Sebelum melihat penyakit, perlu dipahami dulu bagaimana variabel-variabel bebas saling
berkaitan. Kalau dua variabel sangat berkorelasi satu sama lain, keduanya membawa informasi
yang tumpang tindih — dan sebuah variabel bisa tampak berhubungan dengan penyakit padahal
sebenarnya hanya menumpang pada variabel lain.

In [ ]:

# ===================== MATRIKS KORELASI NUMERIK =====================
fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

for ax, metode, judul in [(axes[0], "pearson", "Pearson (hubungan linear)"),
                          (axes[1], "spearman", "Spearman (hubungan monoton)")]:
    sns.heatmap(df[NUM].corr(method=metode), annot=True, fmt=".3f", cmap="coolwarm",
                vmin=-1, vmax=1, square=True, ax=ax, annot_kws={"fontsize": 12},
                cbar_kws={"shrink": 0.8})
    ax.set_title(judul)

fig.suptitle("Korelasi Antar Variabel Numerik — SEMUA USIA (0-113)", y=1.00)
plt.tight_layout()
plt.show()


Tiga hal langsung terlihat, dan **dua di antaranya menyesatkan** kalau dibaca apa adanya:

1. **`Berat Badan` ↔ `IMT` = 0.842** — wajar dan tidak informatif: IMT dihitung *dari* berat
   badan. Keduanya bukan dua informasi terpisah.
2. **`Tinggibadan` ↔ `Berat Badan` = 0.773** — terlihat sangat kuat.
3. **`Usia` ↔ `IMT` = 0.327** — terlihat sedang.

Poin 2 dan 3 dihitung pada **seluruh rentang usia 0–106 tahun**, dan populasi itu memuat
anak-anak. Anak yang bertambah usia akan bertambah tinggi, bertambah berat, dan bertambah IMT
sekaligus — bukan karena ketiganya saling memengaruhi, melainkan karena semuanya mengikuti
**pertumbuhan**.

In [ ]:

# ===================== USIA vs IMT =====================
dewasa = df[df["Usia"] > 20]

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2, sharey=True)

for ax, sub, judul in [(axes[0], df, "SEMUA USIA (0-113)"),
                       (axes[1], dewasa, "DEWASA SAJA (>20)")]:
    hb = ax.hexbin(sub["Usia"], sub["IMT_recalculated"], gridsize=45,
                   cmap="Blues", mincnt=1, bins="log")
    # Median dipakai (bukan rata-rata) supaya tidak tertarik ekor sebaran IMT.
    tren = sub.groupby("Usia")["IMT_recalculated"].median()
    tren = tren[sub.groupby("Usia").size() >= 100]
    ax.plot(tren.index, tren.values, color="crimson", lw=3, label="median IMT per usia")

    r = sub["Usia"].corr(sub["IMT_recalculated"])
    rs = sub["Usia"].corr(sub["IMT_recalculated"], method="spearman")
    ax.set_title(f"{judul}\nPearson r = {r:+.3f}   Spearman = {rs:+.3f}")
    ax.set_xlabel("Usia (tahun)")
    ax.legend(loc="upper right")
    fig.colorbar(hb, ax=ax, label="jumlah (skala log)")

axes[0].set_ylabel("IMT (kg/m2)")
axes[0].set_ylim(10, 45)
fig.suptitle("Usia vs IMT — kepadatan data (hexbin) + garis tren", y=1.00)
plt.tight_layout()
plt.show()


### Temuan: hubungan Usia–IMT **berbalik tanda** saat anak dikeluarkan

| Populasi | Pearson *r* | Tafsiran |
|---|---|---|
| Semua usia (0–106) | **+0.327** | sedang, positif |
| Dewasa saja (>20) | **−0.048** | dapat diabaikan, sedikit negatif |

Angka +0.327 pada seluruh populasi **bukan efek penuaan**, melainkan efek **pertumbuhan**.
Begitu anak-anak dikeluarkan, hubungannya praktis hilang dan bahkan sedikit berbalik — IMT
dewasa cenderung naik sampai usia paruh baya lalu **menurun** pada lansia, terlihat sebagai
lengkungan pada garis merah panel kanan.

Ini juga menjelaskan mengapa Pearson dan Spearman berbeda: hubungannya **tidak lurus**.
Pearson hanya mengukur garis lurus, dan garis lurus adalah model yang keliru di sini.

In [ ]:

# ===================== TINGGI vs BERAT =====================
fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

for ax, sub, judul in [(axes[0], df, "SEMUA USIA (0-113)"),
                       (axes[1], dewasa, "DEWASA SAJA (>20)")]:
    hb = ax.hexbin(sub["Tinggibadan"], sub["Berat Badan"], gridsize=45,
                   cmap="Blues", mincnt=1, bins="log")
    tren = sub.groupby(pd.cut(sub["Tinggibadan"], bins=40), observed=True)["Berat Badan"].median()
    ax.plot([iv.mid for iv in tren.index], tren.values, color="crimson", lw=3,
            label="median berat per tinggi")

    r = sub["Tinggibadan"].corr(sub["Berat Badan"])
    ax.set_title(f"{judul}\nPearson r = {r:+.3f}")
    ax.set_xlabel("Tinggi Badan (cm)")
    ax.set_ylabel("Berat Badan (kg)")
    ax.legend(loc="upper left")
    fig.colorbar(hb, ax=ax, label="jumlah (skala log)")

fig.suptitle("Tinggi vs Berat Badan — kepadatan data (hexbin) + garis tren", y=1.00)
plt.tight_layout()
plt.show()

band = pd.cut(df["Usia"], [0, 2, 5, 10, 15, 20, 40, 60, 120], right=False,
              labels=["0-1", "2-4", "5-9", "10-14", "15-19", "20-39", "40-59", "60+"])
rk = (df.assign(_b=band).groupby("_b", observed=True)
        .apply(lambda g: g["Tinggibadan"].corr(g["Berat Badan"]), include_groups=False))
tab = pd.DataFrame({"n": df.assign(_b=band).groupby("_b", observed=True).size(),
                    "Pearson r": rk.round(3)})
tab["Tafsiran"] = tab["Pearson r"].map(tafsir)
display(tab)


### Temuan: korelasi 0.773 itu sebagian besar efek pertumbuhan

| Populasi | Pearson *r* |
|---|---|
| Semua usia | **0.773** |
| Anak & remaja (2–20) | **0.869** |
| Dewasa (>20) | **0.465** |

Di dalam kelompok dewasa mana pun korelasinya hanya sekitar **0.44–0.50**. Angka gabungan
yang tinggi muncul karena populasinya mencampur bayi 50 cm/4 kg dengan orang dewasa
165 cm/60 kg.

Ini contoh klasik **variabel perancu**: usia memengaruhi tinggi **dan** berat sekaligus,
sehingga menciptakan korelasi antara keduanya yang jauh melebihi hubungan sebenarnya di dalam
kelompok umur yang sebanding.

In [ ]:

# ===================== ETA: KATEGORIK -> NUMERIK (DEWASA) =====================
# Dibatasi ke dewasa supaya efek pertumbuhan tidak mencemari hasilnya.
baris = []
for c in CAT:
    for n in NUM:
        baris.append({"Kategorik": c, "Numerik": n,
                      "eta": round(correlation_ratio(dewasa[c], dewasa[n]), 3)})
eta_tab = pd.DataFrame(baris).pivot(index="Kategorik", columns="Numerik", values="eta")[NUM]

print("Rasio korelasi eta (0..1) — dewasa >20 tahun:")
display(eta_tab)

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

sns.boxplot(data=dewasa, x="Jeniskelamin", y="Tinggibadan", ax=axes[0],
            hue="Jeniskelamin", legend=False, showfliers=False,
            palette=vz.PALETTE["SEX"])
axes[0].set_title(f"Tinggi Badan menurut Jenis Kelamin\n"
                  f"(eta = {eta_tab.loc['Jeniskelamin','Tinggibadan']:.3f})")

urut = dewasa.groupby("segmen")["Usia"].median().sort_values().index
sns.boxplot(data=dewasa, x="segmen", y="Usia", order=urut, ax=axes[1],
            color="steelblue", showfliers=False)
axes[1].set_title(f"Usia menurut Segmen\n(eta = {eta_tab.loc['segmen','Usia']:.3f})")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


### Temuan: dua perancu yang akan muncul lagi nanti

- **Jenis kelamin → Tinggi badan, η ≈ 0.58.** Kuat dan sepenuhnya terduga secara biologis.
  Konsekuensinya: `Tinggibadan` sebagian berfungsi sebagai **penanda jenis kelamin**. Ini
  ditagih kembali di §6 (Kanker Paru).
- **Segmen → Usia, η ≈ 0.49.** Segmen kepesertaan BPJS sangat berkaitan dengan tahap hidup —
  misalnya segmen "bukan pekerja" banyak berisi pensiunan. Ini ditagih kembali di §7 (TBC).

Keduanya berarti: bila nanti tinggi badan atau segmen tampak berhubungan dengan penyakit,
sebagian hubungan itu bisa jadi milik jenis kelamin atau usia.


---
## Fungsi penggambar profil penyakit

Satu fungsi dipakai untuk keempat penyakit agar §4–§7 tidak mengulang kode dan seluruh
gambar dijamin memakai perlakuan yang sama persis.

In [ ]:

# ===================== FUNGSI PROFIL PENYAKIT =====================
def profil_penyakit(nama, n_desil=10):
    # Menggambar satu gambar berisi 9 panel:
    #   baris 1 (4 panel) : histogram densitas Berisiko vs Tidak Berisiko
    #   baris 2 (4 panel) : garis prevalensi per desil
    #   kolom 5           : batang prevalensi menurut jenis kelamin (membentang 2 baris)
    # Mengembalikan tabel ringkas asosiasi untuk penyakit ini.
    sub, y = eligible(nama)
    base = y.mean()
    usia_lbl = TARGETS[nama][1]

    fig = plt.figure(figsize=(27, 11))
    gs = gridspec.GridSpec(2, 5, figure=fig, width_ratios=[1, 1, 1, 1, 0.7],
                           hspace=0.32, wspace=0.28)

    hasil = {}
    for j, var in enumerate(NUM):
        # ---------- baris 1: histogram densitas ----------
        ax_h = fig.add_subplot(gs[0, j])
        lo, hi = sub[var].quantile([0.005, 0.995])      # potong ekor agar terbaca
        bins = np.linspace(lo, hi, 45)
        for kelas, lbl in [(0, "Tidak Berisiko"), (1, "Berisiko")]:
            ax_h.hist(sub.loc[y == kelas, var], bins=bins, density=True, alpha=0.55,
                      color=vz.PALETTE["RISK"][lbl], label=lbl)
        ax_h.set_title(var)
        ax_h.set_ylabel("kepadatan" if j == 0 else "")
        ax_h.set_xlim(lo, hi)
        if j == 0:
            ax_h.legend(fontsize=9)

        # ---------- baris 2: garis prevalensi per desil ----------
        ax_l = fig.add_subplot(gs[1, j])
        q = pd.qcut(sub[var], n_desil, duplicates="drop")
        prev = y.groupby(q, observed=True).mean() * 100
        # Titik diletakkan pada MEDIAN nilai di dalam tiap desil, bukan titik tengah
        # intervalnya. Desil pertama dan terakhir intervalnya sangat lebar (menjangkau
        # nilai ekstrem), sehingga titik tengahnya bisa jatuh di luar sumbu.
        xpos = sub.groupby(q, observed=True)[var].median()
        ax_l.plot(xpos.values, prev.values, marker="o", lw=2.5, color="#153763")
        ax_l.axhline(base * 100, color="crimson", ls="--",
                     label=f"rata-rata {base*100:.1f}%")
        r = sub[var].corr(y)
        hasil[var] = r
        ax_l.set_title(f"r = {r:+.3f}  [{tafsir(r)}]", fontsize=12)
        ax_l.set_xlabel(f"{var} (titik tengah desil)")
        ax_l.set_ylabel("% Berisiko" if j == 0 else "")
        ax_l.set_xlim(lo, hi)                            # sejajar dengan histogram di atas
        ax_l.legend(fontsize=9)

    # ---------- kolom 5: jenis kelamin ----------
    ax_s = fig.add_subplot(gs[:, 4])
    prev_s = y.groupby(sub["Jeniskelamin"], observed=True).mean() * 100
    ax_s.bar(prev_s.index, prev_s.values,
             color=[vz.PALETTE["SEX"][s] for s in prev_s.index])
    ax_s.axhline(base * 100, color="crimson", ls="--", label=f"rata-rata {base*100:.1f}%")
    v = cramers_v(sub["Jeniskelamin"], y)
    hasil["Jeniskelamin"] = v
    ax_s.set_title(f"Jenis Kelamin\nCramer's V = {v:.3f}  [{tafsir(v)}]")
    ax_s.set_ylabel("% Berisiko")
    ax_s.legend(fontsize=9)
    vz.annotate_bars(ax_s, fmt="{:.1f}%")

    lbl_usia = usia_lbl if usia_lbl == "semua usia" else f"usia {usia_lbl}"
    fig.suptitle(f"{nama}  —  {lbl_usia}  |  n = {len(sub):,}  |  "
                 f"prevalensi {base*100:.2f}%", y=0.99, fontsize=17)
    plt.show()

    ringkas = pd.DataFrame({"Ukuran": pd.Series(hasil).round(3)})
    ringkas["Tafsiran"] = ringkas["Ukuran"].map(tafsir)
    ringkas["Jenis ukuran"] = ["point-biserial r"] * len(NUM) + ["Cramer's V"]
    return ringkas.reindex(NUM + ["Jeniskelamin"])


print("Fungsi profil_penyakit() siap.")


---
# 4. Diabetes Melitus

Berlaku untuk usia **15 tahun ke atas**.

In [ ]:

display(profil_penyakit("Diabetes"))


### Membaca profil Diabetes

**Baris atas** memperlihatkan pemisahan paling jelas di antara keempat penyakit: pada panel
`IMT_recalculated` dan `Berat Badan`, kurva merah (Berisiko) dan hijau (Tidak Berisiko)
hampir tidak bertumpang tindih. Kedua kelompok benar-benar menempati wilayah yang berbeda.

**Baris bawah** memperlihatkan bentuk yang tidak biasa pada IMT: bukan kenaikan bertahap,
melainkan **lompatan tegak** di sekitar IMT 25, dari belasan persen langsung ke hampir 100%.

| Variabel | *r* | Tafsiran |
|---|---|---|
| IMT | **+0.630** | kuat |
| Berat Badan | +0.538 | kuat |
| Usia | +0.242 | lemah |
| Tinggi Badan | −0.048 | dapat diabaikan |
| Jenis Kelamin | 0.037 | dapat diabaikan |

### Peringatan: angka 0.630 bukan temuan epidemiologis

Label "Berisiko Diabetes" pada kuesioner skrining praktis **diturunkan dari aturan
`IMT > 25`** — `P(Berisiko | IMT > 25) = 0.9985`. Bentuk tangga pada grafik itu adalah tanda
aturan, bukan tanda biologi.

Jadi korelasi kuat antara IMT dan label diabetes bersifat **melingkar**: kita mengukur ulang
aturan yang dipakai untuk membuat labelnya. Buktinya lengkap ada di `Analisis Data Mentah` §8
dan `Klasifikasi BMI` §7.

Yang tersisa sebagai informasi nyata adalah **usia** (+0.242): di antara orang ber-IMT di
bawah 25, proporsi berisiko tetap naik seiring usia.


---
# 5. Hipertensi

Berlaku untuk usia **15 tahun ke atas**.

In [ ]:

display(profil_penyakit("Hipertensi"))


### Membaca profil Hipertensi

Polanya mirip Diabetes tetapi **jauh lebih lemah**, dan bentuknya berbeda secara penting.

| Variabel | *r* | Tafsiran |
|---|---|---|
| IMT | **+0.328** | sedang |
| Berat Badan | +0.314 | sedang |
| Usia | +0.055 | dapat diabaikan |
| Tinggi Badan | +0.052 | dapat diabaikan |
| Jenis Kelamin | 0.050 | dapat diabaikan |

**Perbedaan bentuk itu penting.** Grafik IMT pada Hipertensi menanjak **mulus dan bertahap**
di sepanjang rentang — tidak ada lompatan di titik tertentu. Inilah bentuk yang diharapkan
dari hubungan biologis yang sesungguhnya, dan kontras jelas dengan bentuk tangga milik
Diabetes.

Dengan kata lain: Hipertensi memberi bukti bahwa IMT **memang** berkaitan dengan risiko
kesehatan — dan justru karena bentuknya mulus, hubungan ini lebih dapat dipercaya daripada
angka Diabetes yang jauh lebih besar.

**Usia hampir tidak berpengaruh** (+0.055), meski prevalensinya tinggi (62,4%). Hipertensi
banyak ditemukan di semua kelompok usia dewasa, bukan khusus pada lansia. Pada baris atas
terlihat kedua kurva usia hampir bertumpuk sempurna.


---
# 6. Kanker Paru

Berlaku untuk usia **45 tahun ke atas**. Perhatikan bahwa rentang usianya sudah terpotong,
sehingga variasi usia yang tersisa kecil.

In [ ]:

display(profil_penyakit("Kanker Paru"))


### Membaca profil Kanker Paru

Profil penyakit ini **sepenuhnya berbeda** dari dua sebelumnya.

| Variabel | Ukuran | Tafsiran |
|---|---|---|
| **Jenis Kelamin** | **0.285** | **lemah — tertinggi** |
| Tinggi Badan | +0.185 | lemah |
| Berat Badan | +0.097 | dapat diabaikan |
| IMT | −0.002 | dapat diabaikan |
| Usia | −0.036 | dapat diabaikan |

Panel jenis kelamin di kolom kanan langsung menjelaskan semuanya: **19,9% pada laki-laki
melawan 2,7% pada perempuan** — selisih sekitar tujuh kali lipat.

IMT dan usia praktis tidak berpengaruh; pada baris atas, kurva merah dan hijau untuk kedua
variabel itu hampir bertumpuk sempurna.

Tetapi ada satu angka yang mencurigakan: **Tinggi Badan +0.185**. Secara biologis tidak ada
alasan tinggi badan memengaruhi risiko kanker paru. Bagian berikut mengujinya.

In [ ]:

# ===================== UJI: APAKAH TINGGI BADAN HANYA PROKSI JENIS KELAMIN? =====================
sub_k, y_k = eligible("Kanker Paru")
sub_k = sub_k.assign(_y=y_k.values)

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

# --- Kiri: prevalensi vs tinggi, gabungan dan dipisah jenis kelamin ---
q_all = pd.qcut(sub_k["Tinggibadan"], 10, duplicates="drop")
prev_all = sub_k.groupby(q_all, observed=True)["_y"].mean() * 100
x_all = sub_k.groupby(q_all, observed=True)["Tinggibadan"].median()
axes[0].plot(x_all.values, prev_all.values, marker="o", lw=3,
             color="black", label="gabungan (L+P)")

for s, lbl in [("L", "Laki-laki saja"), ("P", "Perempuan saja")]:
    g = sub_k[sub_k["Jeniskelamin"] == s]
    q = pd.qcut(g["Tinggibadan"], 10, duplicates="drop")
    prev = g.groupby(q, observed=True)["_y"].mean() * 100
    xg = g.groupby(q, observed=True)["Tinggibadan"].median()
    axes[0].plot(xg.values, prev.values, marker="o", lw=2.2,
                 color=vz.PALETTE["SEX"][s], label=lbl)

axes[0].set_title("Kanker Paru vs Tinggi Badan\ngabungan vs di dalam tiap jenis kelamin")
axes[0].set_xlabel("Tinggi Badan (titik tengah desil)")
axes[0].set_ylabel("% Berisiko")
axes[0].legend()

# --- Kanan: korelasi sebelum vs sesudah dikendalikan ---
korelasi = {"Gabungan\n(L+P)": sub_k["Tinggibadan"].corr(sub_k["_y"])}
for s, lbl in [("L", "Laki-laki\nsaja"), ("P", "Perempuan\nsaja")]:
    g = sub_k[sub_k["Jeniskelamin"] == s]
    korelasi[lbl] = g["Tinggibadan"].corr(g["_y"])

axes[1].bar(list(korelasi), list(korelasi.values()),
            color=["black", vz.PALETTE["SEX"]["L"], vz.PALETTE["SEX"]["P"]])
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Korelasi Tinggi Badan - Kanker Paru")
axes[1].set_ylabel("point-biserial r")
vz.annotate_bars(axes[1], fmt="{:+.3f}")

plt.tight_layout()
plt.show()

for k, v in korelasi.items():
    print(f"  {k.replace(chr(10), ' '):<22} r = {v:+.3f}")


### Temuan: hubungan Tinggi Badan–Kanker Paru **sepenuhnya** efek jenis kelamin

| Populasi | point-biserial *r* |
|---|---|
| Gabungan (L+P) | **+0.185** |
| Laki-laki saja | **+0.052** |
| Perempuan saja | **−0.008** |

Begitu jenis kelamin dikendalikan, hubungannya **lenyap**. Pada panel kiri terlihat jelas:
garis hitam (gabungan) menanjak, tetapi garis biru dan merah muda masing-masing **datar** —
dan keduanya berada pada ketinggian yang sangat berbeda.

Mekanismenya sederhana: laki-laki rata-rata lebih tinggi (η = 0.58 dari §3), dan laki-laki
punya risiko Kanker Paru jauh lebih besar. Tinggi badan hanya **menandai** jenis kelamin.

Ini contoh perancu paling bersih di seluruh notebook: korelasi 0.185 yang terlihat cukup
berarti ternyata **nol** setelah dikoreksi.

**Konsekuensi untuk laporan:** jangan pernah menulis bahwa orang yang lebih tinggi berisiko
Kanker Paru lebih besar. Yang benar: laki-laki berisiko lebih besar, dan laki-laki kebetulan
lebih tinggi. Jenis kelamin sendiri kemungkinan besar adalah **penanda pengganti kebiasaan
merokok**, yang tidak tersedia di data.


---
# 7. TBC

Berlaku untuk **semua usia (0–106)** — satu-satunya target tanpa batas usia. Justru karena
itu, TBC paling rawan terkena efek perancu usia.

In [ ]:

display(profil_penyakit("TBC"))


### Membaca profil TBC

| Variabel | Ukuran | Tafsiran |
|---|---|---|
| Usia | **+0.125** | lemah — tertinggi |
| Berat Badan | +0.068 | dapat diabaikan |
| IMT | +0.067 | dapat diabaikan |
| Tinggi Badan | +0.042 | dapat diabaikan |
| Jenis Kelamin | 0.009 | dapat diabaikan |

**Seluruh variabel lemah.** Nilai tertinggi hanya 0.125, dan sisanya di bawah ambang 0.10.
Tidak ada satu pun karakteristik yang menjelaskan risiko TBC dengan baik.

Pada baris atas, kurva merah dan hijau bertumpang tindih sangat besar di keempat variabel —
gambaran visual dari kesimpulan yang sama. Ini konsisten dengan performa model yang paling
rendah pada target ini: keterbatasannya ada pada **variabel yang tersedia**, bukan pada
metode.

Dua hubungan tetap perlu diuji karena TBC mencakup semua usia:

1. **TBC ↔ IMT** — grafik naik, padahal literatur justru mengaitkan TBC dengan gizi kurang
2. **TBC ↔ segmen** — segmen sangat terkait usia (η = 0.49 dari §3)

In [ ]:

# ===================== UJI PERANCU 1: TBC vs IMT, DIKENDALIKAN USIA =====================
sub_b, y_b = eligible("TBC")
sub_b = sub_b.assign(_y=y_b.values)
band_b = pd.cut(sub_b["Usia"], [0, 15, 30, 45, 60, 120], right=False,
                labels=["0-14", "15-29", "30-44", "45-59", "60+"])
sub_b = sub_b.assign(_band=band_b)

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

for b in sub_b["_band"].cat.categories:
    s = sub_b[sub_b["_band"] == b]
    if len(s) < 500:
        continue
    q = pd.qcut(s["IMT_recalculated"], 5, duplicates="drop")
    prev = s.groupby(q, observed=True)["_y"].mean() * 100
    xq = s.groupby(q, observed=True)["IMT_recalculated"].median()
    axes[0].plot(xq.values, prev.values, marker="o", label=f"usia {b}")

axes[0].set_title("TBC vs IMT — dipisah per kelompok usia")
axes[0].set_xlabel("IMT (titik tengah kuintil)")
axes[0].set_ylabel("% Berisiko TBC")
axes[0].legend(title="Kelompok usia")

kes = sub_b["IMT_recalculated"].corr(sub_b["_y"])
dalam = (sub_b.groupby("_band", observed=True)
              .apply(lambda g: g["IMT_recalculated"].corr(g["_y"]), include_groups=False))
axes[1].bar(dalam.index.astype(str), dalam.values, color="steelblue")
axes[1].axhline(kes, color="crimson", ls="--", label=f"gabungan semua usia ({kes:+.3f})")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Korelasi TBC-IMT: gabungan vs di dalam kelompok usia")
axes[1].set_xlabel("Kelompok usia")
axes[1].set_ylabel("point-biserial r")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Korelasi TBC-IMT gabungan semua usia : {kes:+.3f}")
for b, v in dalam.items():
    print(f"   usia {str(b):<7} {v:+.3f}")


### Temuan 1: sebagian efek usia, tetapi tidak seluruhnya

| Populasi | *r* |
|---|---|
| Gabungan semua usia | **+0.067** |
| Usia 0–14 | **−0.015** (berbalik tanda) |
| Usia 15–29 | +0.021 |
| Usia 30–44 | +0.050 |
| Usia 45–59 | +0.062 |
| Usia 60+ | **+0.073** |

Pada **anak-anak hubungannya berbalik**, pada **dewasa muda praktis nol**, tetapi pada
**lansia nilainya bahkan sedikit melampaui angka gabungan**.

Jadi kesimpulan yang jujur bukan "seluruhnya efek usia", melainkan: sebagian kenaikan pada
grafik profil memang efek komposisi usia, tetapi pada kelompok dewasa tua tetap ada hubungan
positif lemah yang bertahan.

**Yang terpenting: seluruh nilainya dapat diabaikan.** Bahkan yang tertinggi (0.073) jauh di
bawah ambang 0.10. IMT bukan penjelas risiko TBC yang berarti.

In [ ]:

# ===================== UJI PERANCU 2: TBC vs SEGMEN, DIKENDALIKAN USIA =====================
piv = sub_b.groupby(["_band", "segmen"], observed=True)["_y"].agg(["mean", "size"])
piv = piv[piv["size"] >= 300]["mean"].unstack() * 100

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
sns.heatmap(piv, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax,
            annot_kws={"fontsize": 11}, cbar_kws={"label": "% Berisiko TBC"})
ax.set_title("Prevalensi TBC menurut Segmen DI DALAM Tiap Kelompok Usia\n"
             "(sel dengan n < 300 dikosongkan)")
ax.set_xlabel("Segmen")
ax.set_ylabel("Kelompok usia")
plt.tight_layout()
plt.show()

print(f"Cramer's V segmen-TBC, gabungan semua usia : {cramers_v(sub_b['segmen'], sub_b['_y']):.3f}")
for b in sub_b["_band"].cat.categories:
    s = sub_b[sub_b["_band"] == b]
    if len(s) >= 500:
        print(f"   usia {str(b):<7} {cramers_v(s['segmen'], s['_y']):.3f}")


### Temuan 2: hubungan segmen–TBC **bertahan**, dan justru menguat pada usia tua

| Populasi | Cramér's V |
|---|---|
| Gabungan semua usia | **0.099** |
| Usia 0–14 | 0.039 |
| Usia 15–29 | 0.030 |
| Usia 30–44 | 0.039 |
| Usia 45–59 | 0.059 |
| Usia 60+ | **0.098** |

Asosiasi segmen–TBC **menguat seiring usia**, dan pada kelompok 60+ nilainya praktis sama dengan
angka gabungan. Artinya hubungan ini **tidak runtuh** ketika usia dikendalikan — berbeda
dari IMT yang berbalik tanda pada anak-anak.

Segmen membawa informasi di luar sekadar usia, tetapi informasinya **hanya relevan pada
kelompok dewasa tua**. Karena segmen mencerminkan kategori kepesertaan BPJS — dan secara
tidak langsung kondisi sosial ekonomi — arah temuan ini sejalan dengan literatur yang
mengaitkan risiko TBC dengan faktor sosial ekonomi.

**Tetap perlu kejujuran soal besaran:** 0.098 masih tergolong "dapat diabaikan". Segmen
adalah variabel terkuat kedua untuk TBC, tetapi itu lebih menunjukkan **betapa lemahnya semua
variabel untuk TBC** daripada betapa kuatnya segmen.


---
# 8. Segmen: Semua Penyakit

`segmen` adalah kategori kepesertaan BPJS, dan merupakan **satu-satunya proksi sosial
ekonomi** yang tersedia di data ini:

| Kode | Arti |
|---|---|
| APBN / APBD | Iuran ditanggung pemerintah pusat / daerah |
| PPU PN / PPU BU | Pekerja penerima upah — penyelenggara negara / badan usaha |
| PBPU | Pekerja bukan penerima upah (mandiri) |
| BP PN / BP SWASTA | Bukan pekerja (termasuk pensiunan) |

Karena tidak punya urutan alami, segmen tidak bisa digambar sebagai garis tren. Bentuk yang
tepat adalah **peta panas**: seluruh penyakit sekaligus, sehingga polanya bisa dibandingkan.

In [ ]:

# ===================== PETA PANAS SEGMEN x PENYAKIT =====================
prev_sg, rr_sg, v_sg = {}, {}, {}
for nama in TARGETS:
    sub, y = eligible(nama)
    p = y.groupby(sub["segmen"], observed=True).mean() * 100
    prev_sg[nama] = p
    rr_sg[nama] = p / (y.mean() * 100)          # risiko relatif terhadap rata-rata
    v_sg[nama] = cramers_v(sub["segmen"], y)

prev_sg = pd.DataFrame(prev_sg)
rr_sg = pd.DataFrame(rr_sg)

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

sns.heatmap(prev_sg, annot=True, fmt=".1f", cmap="YlOrRd", ax=axes[0],
            annot_kws={"fontsize": 11}, cbar_kws={"label": "% Berisiko"}, linewidths=0.5)
axes[0].set_title("Prevalensi mentah (%)\nbagus untuk beban layanan")
axes[0].set_ylabel("")

sns.heatmap(rr_sg, annot=True, fmt=".2f", cmap="RdYlGn_r", center=1.0, ax=axes[1],
            annot_kws={"fontsize": 11}, cbar_kws={"label": "risiko relatif"}, linewidths=0.5)
axes[1].set_title("Risiko relatif (1,00 = rata-rata penyakit itu)\nbagus untuk perbandingan")
axes[1].set_ylabel("")

fig.suptitle("Segmen Kepesertaan vs Keempat Risiko Penyakit", y=1.00)
plt.tight_layout()
plt.show()

print("Kekuatan asosiasi segmen dengan tiap penyakit (Cramer's V):")
for nama, v in v_sg.items():
    print(f"  {nama:<13} V = {v:.3f}  [{tafsir(v)}]")
print("\nRentang risiko relatif (tertinggi : terendah):")
for nama in TARGETS:
    s = rr_sg[nama]
    print(f"  {nama:<13} {s.max()/s.min():.2f}x   "
          f"(tertinggi {s.idxmax()} {s.max():.2f}x, terendah {s.idxmin()} {s.min():.2f}x)")


### Kenapa dua peta panas

**Peta kiri (prevalensi mentah)** menjawab *"berapa persen peserta di segmen ini yang
berisiko"*. Berguna untuk perencanaan beban layanan, tetapi tidak bisa dibandingkan antar
penyakit — kolom Hipertensi akan selalu terlihat merah karena prevalensi dasarnya memang 62%.

**Peta kanan (risiko relatif)** membagi tiap nilai dengan prevalensi penyakitnya sendiri,
sehingga **1,00 berarti sama dengan rata-rata** penyakit itu. Sekarang keempat kolom bisa
dibandingkan langsung: hijau = di bawah rata-rata, merah = di atas.

### Temuan

**TBC menunjukkan gradien paling lebar** — rentangnya sekitar 3,0 kali lipat antara segmen
tertinggi dan terendah, jauh melampaui tiga penyakit lain yang semuanya di bawah 1,7 kali.
Ini sejalan dengan §7: segmen adalah variabel terkuat kedua untuk TBC dan bertahan setelah
usia dikendalikan.

**Hipertensi paling datar** (rentang 1,11x) — seluruh segmen praktis setara.

Perlu diingat dari §3 bahwa segmen sangat terkait usia (η = 0.49), sehingga sebagian pola di
sini adalah efek usia. Untuk TBC hal itu sudah diuji secara khusus di §7 dan hubungannya
bertahan; untuk penyakit lain gradiennya terlalu kecil untuk ditarik kesimpulan.


---
# 9. Ringkasan Lintas Penyakit

In [ ]:

# ===================== MATRIKS ASOSIASI LENGKAP =====================
baris = []
for nama in TARGETS:
    sub, y = eligible(nama)
    r = {"Target": nama}
    for c in NUM:
        r[c] = sub[c].corr(y)                 # point-biserial
    for c in CAT:
        r[c] = cramers_v(sub[c], y)           # phi / Cramer's V
    baris.append(r)

asosiasi = pd.DataFrame(baris).set_index("Target")

print("Numerik = point-biserial r (bertanda).  Kategorik = Cramer's V (0..1, tanpa arah).")
display(asosiasi.round(3))

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
sns.heatmap(asosiasi.abs(), annot=asosiasi.round(3), fmt="", cmap="YlOrRd",
            vmin=0, vmax=0.65, ax=ax, annot_kws={"fontsize": 12},
            cbar_kws={"label": "kekuatan asosiasi (nilai mutlak)"}, linewidths=0.5)
ax.set_title("Kekuatan Hubungan Tiap Variabel dengan Tiap Risiko Penyakit\n"
             "(warna = kekuatan, angka = nilai asli beserta tandanya)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:

# ===================== VARIABEL TERKUAT PER PENYAKIT =====================
ring = []
for nama in TARGETS:
    s = asosiasi.loc[nama]
    urut = s.abs().sort_values(ascending=False)
    ring.append({
        "Target": nama,
        "Rentang usia": TARGETS[nama][1],
        "Terkuat": urut.index[0], "Nilai": round(s[urut.index[0]], 3),
        "Tafsiran": tafsir(s[urut.index[0]]),
        "Kedua": urut.index[1], "Nilai 2": round(s[urut.index[1]], 3),
        "Ketiga": urut.index[2], "Nilai 3": round(s[urut.index[2]], 3),
    })

display(pd.DataFrame(ring).set_index("Target"))


## Jawaban ringkas atas pertanyaan awal

| Pertanyaan | Jawaban |
|---|---|
| **Apakah usia berhubungan dengan IMT?** | Ya bila semua usia digabung (*r* = +0.327), tetapi itu **efek pertumbuhan**. Pada dewasa saja praktis nol (*r* = −0.048) dan berbentuk lengkung |
| **Apakah usia berhubungan dengan risiko penyakit?** | Berbeda-beda. Paling jelas pada Diabetes (0.242) dan TBC (0.125); **hampir tidak ada** pada Hipertensi (0.055) dan Kanker Paru (−0.036) |
| **Apakah IMT berhubungan dengan risiko penyakit?** | Kuat pada Diabetes (0.630) — tetapi **melingkar**. Sedang pada Hipertensi (0.328) dengan bentuk mulus yang lebih dapat dipercaya. **Dapat diabaikan** pada Kanker Paru dan TBC |
| **Bagaimana tinggi dan berat berhubungan?** | Positif dan nyata, tetapi jauh lebih lemah daripada kesan awal: 0.773 semua usia, hanya **0.465** pada dewasa |
| **Apakah jenis kelamin atau segmen bermakna?** | **Jenis kelamin paling bermakna untuk Kanker Paru saja** (V = 0.285). **Segmen paling bermakna untuk TBC** (V = 0.099) dan bertahan setelah usia dikendalikan |
| **Variabel mana yang terkuat untuk tiap penyakit?** | Diabetes → **IMT**; Hipertensi → **IMT**; Kanker Paru → **Jenis kelamin**; TBC → **Usia**, disusul **segmen** |

## Empat pola yang layak dibawa ke laporan

1. **Keempat penyakit punya profil yang berbeda.** Tidak ada satu variabel yang menjelaskan
   semuanya. Diabetes dan Hipertensi digerakkan komposisi tubuh, Kanker Paru oleh jenis
   kelamin, TBC oleh usia dan kondisi sosial ekonomi.

2. **Hubungan terkuat justru yang paling tidak informatif.** IMT ↔ Diabetes (0.630) adalah
   yang tertinggi di seluruh analisis, tetapi melingkar. Yang lebih layak dilaporkan justru
   IMT ↔ Hipertensi (0.328), karena bentuknya mulus dan tidak berasal dari aturan.

3. **TBC hampir tidak bisa dijelaskan oleh data yang tersedia.** Asosiasi tertinggi hanya
   0.125. Keterbatasannya ada pada **variabel yang tersedia**, bukan pada metode.

4. **Dari tiga hubungan yang tampak kuat, dua ternyata palsu.** Tinggi ↔ Kanker Paru
   sepenuhnya efek jenis kelamin (§6); IMT ↔ TBC sebagian efek usia (§7). Uji perancu bukan
   pelengkap — justru itu yang menentukan temuan mana yang boleh dilaporkan.

## Peringatan perbandingan

Kekuatan asosiasi **tidak sepenuhnya sebanding antar penyakit** karena populasinya berbeda.
Kanker Paru hanya mencakup usia 45+, sehingga variasi usianya kecil dan korelasi usianya
otomatis mengecil. TBC mencakup semua usia sehingga justru paling rawan perancu usia.


---
# 10. Korelasi Bukan Sebab-Akibat

Seluruh angka dalam notebook ini adalah **asosiasi**: dua hal cenderung muncul bersamaan.
Itu **tidak** membuktikan bahwa yang satu menyebabkan yang lain. Setidaknya ada empat
penjelasan lain untuk setiap hubungan di atas.

**1. Variabel perancu.** Sesuatu yang lain memengaruhi keduanya sekaligus. Sudah terbukti
tiga kali di notebook ini:

| Hubungan yang tampak | Setelah dikendalikan | Perancunya |
|---|---|---|
| Tinggi ↔ Berat = 0.773 | 0.465 | usia |
| Tinggi ↔ Kanker Paru = 0.185 | ~0.00 | jenis kelamin |
| IMT ↔ TBC = +0.067 | −0.015 s.d. +0.073 | usia |

**2. Arah sebab yang terbalik.** Korelasi tidak punya arah. Penyakit yang menyebabkan
penurunan berat badan akan terlihat sama saja dengan berat badan rendah yang menyebabkan
penyakit.

**3. Label yang bersifat aturan.** Kasus Diabetes adalah contoh ekstrem: labelnya *dihitung*
dari IMT. Korelasi 0.630 itu mengukur aturan penilaian kuesioner, bukan biologi.

**4. Bias partisipasi.** Data ini hanya memuat orang yang **ikut skrining**, bukan seluruh
penduduk Sleman. Orang yang merasa dirinya berisiko mungkin lebih terdorong ikut.

## Batasan tambahan

- **Semua angka berlaku per catatan skrining, bukan per orang.** Kolom `id` sengaja diacak
  untuk melindungi data asli BPJS, sehingga tidak mungkin memastikan satu orang hanya
  terhitung sekali.
- **Faktor risiko paling penting justru tidak tersedia**: status merokok, riwayat keluarga,
  lingkar perut, tekanan darah, dan aktivitas fisik. Jenis kelamin pada Kanker Paru hampir
  pasti hanyalah **penanda pengganti** untuk kebiasaan merokok.
- **Nilai-p sengaja tidak dilaporkan.** Pada n = 410.398, hubungan sekecil apa pun akan
  "signifikan"; yang relevan adalah besaran efeknya.

Dengan data yang ada, kesimpulan yang jujur berbentuk *"kelompok ini memiliki risiko lebih
tinggi"*, bukan *"karakteristik ini menyebabkan penyakit"*.